# Demo backend: export the classifier for CPU serving (onnx-export)

**Settings:** Accelerator **None (CPU)**, Internet **ON**. **Input:** **llm-cls-train-v2**. Run as a commit (~40 min).

Goal: run our classifier on a **free CPU** (Hugging Face Space). We try 4 CPU engines on the same rows and keep the fastest one that stays accurate:

| Engine | What |
|---|---|
| torch fp32 | Plain PyTorch on CPU (reference) |
| torch int8 | PyTorch dynamic INT8 quantization of all Linear layers |
| onnx fp32 | Exported to ONNX, run with ONNX Runtime |
| onnx int8 | ONNX Runtime dynamic INT8 quantization |

Output: `cpu_engines.csv` (accuracy + latency per engine) and `hf_model/` (the chosen model + tokenizer + config), ready to upload to the Hugging Face Hub.

In [ ]:
# keep huggingface_hub < 2.0: transformers 5.16 refuses to import with 2.x
!pip install -q -U onnx onnxruntime onnxscript "huggingface_hub>=1.5,<2" 2>&1 | tail -n 2
!pip list 2>/dev/null | grep -E "^(torch|transformers|huggingface-hub|onnx|onnxruntime|onnxscript) "

In [ ]:
import os, gc, glob, json, time, shutil, traceback
import numpy as np, pandas as pd, torch
from sklearn.metrics import log_loss
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def find(p):
    hits = glob.glob(f"/kaggle/input/**/{p}", recursive=True)
    assert hits, f"input not found: {p} (Add Input llm-cls-train-v2?)"
    return hits[0]

MODEL_DIR = os.path.dirname(find("qwen-cls/config.json"))
INFO = json.load(open(find("train_info.json")))
OUT = "/kaggle/working"; ONNX_DIR = f"{OUT}/onnx"; os.makedirs(ONNX_DIR, exist_ok=True)
N_EVAL = 80                                    # val rows per engine, batch size 1 (= the demo case)
THREADS = os.cpu_count()
torch.set_num_threads(THREADS)
print("CPU threads:", THREADS, "| train info:", INFO)

tok = AutoTokenizer.from_pretrained(MODEL_DIR)
HEADERS = dict(prompt="Prompt: ", a="\n\nResponse A: ", b="\n\nResponse B: ", q="\n\nWhich response is better?")
H = {k: tok(v, add_special_tokens=False)["input_ids"] for k, v in HEADERS.items()}
def encode(p, a, b):
    T = lambda s, n: tok(s, add_special_tokens=False, truncation=True, max_length=n)["input_ids"]
    return H["prompt"] + T(p, INFO["max_p"]) + H["a"] + T(a, INFO["max_r"]) + H["b"] + T(b, INFO["max_r"]) + H["q"]

val = pd.read_parquet(find("val.parquet")).iloc[:N_EVAL]
ROWS = [encode(p, a, b) for p, a, b in zip(val.prompt, val.response_a, val.response_b)]
Y = val["label"].values
print("eval rows:", len(ROWS), "| mean tokens:", int(np.mean([len(r) for r in ROWS])))

RESULTS = []
def evaluate(name, run_one):
    run_one(ROWS[0])                           # warmup
    probs, ms = [], []
    for ids in ROWS:
        t = time.perf_counter(); probs.append(run_one(ids)); ms.append((time.perf_counter() - t) * 1000)
    probs = np.array(probs, dtype=np.float32)
    r = dict(engine=name, rows=len(ROWS), p50_ms=round(float(np.median(ms)), 0), p95_ms=round(float(np.percentile(ms, 95)), 0),
             log_loss=round(log_loss(Y, probs, labels=[0, 1, 2]), 4), acc=round(float((probs.argmax(1) == Y).mean()), 4))
    RESULTS.append(r); print(r)
    return probs

## 1. PyTorch fp32 on CPU (reference) + ONNX export

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, dtype=torch.float32,
                                                           attn_implementation="eager").eval()

class ProbsWrapper(torch.nn.Module):                # export the softmax too, so the demo gets probabilities directly
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask):
        return torch.softmax(self.m(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).logits.float(), -1)
wrapped = ProbsWrapper(model).eval()

@torch.inference_mode()
def torch_one(ids):
    x = torch.tensor([ids]); return wrapped(x, torch.ones_like(x))[0].numpy()

P_REF = evaluate("torch_fp32", torch_one)

ONNX_FP32 = f"{ONNX_DIR}/model_fp32.onnx"
EXPORT = {"ok": False}
ex_len = min(len(ROWS[0]), len(ROWS[1]))           # batch of 2 so the batch axis stays dynamic
ex_ids = torch.tensor([ROWS[0][:ex_len], ROWS[1][:ex_len]]); ex_mask = torch.ones_like(ex_ids)
for method in ("dynamo", "legacy"):
    try:
        t = time.time()
        with torch.inference_mode():
            if method == "dynamo":
                batch, seq = torch.export.Dim("batch", min=1, max=16), torch.export.Dim("seq", min=8, max=1024)
                prog = torch.onnx.export(wrapped, (ex_ids, ex_mask), dynamo=True,
                                         input_names=["input_ids", "attention_mask"], output_names=["probs"],
                                         dynamic_shapes={"input_ids": {0: batch, 1: seq}, "attention_mask": {0: batch, 1: seq}})
                prog.save(ONNX_FP32, external_data=True)
            else:
                torch.onnx.export(wrapped, (ex_ids, ex_mask), ONNX_FP32, dynamo=False, opset_version=17,
                                  input_names=["input_ids", "attention_mask"], output_names=["probs"],
                                  dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"},
                                                "probs": {0: "batch"}})
        EXPORT = {"ok": True, "method": method, "seconds": round(time.time() - t, 1)}
        break
    except Exception as e:
        traceback.print_exc()
        EXPORT[method] = repr(e)[:300]
print("EXPORT:", EXPORT)
!ls -la {ONNX_DIR} | head

## 2. PyTorch dynamic INT8 (fallback engine that always works)

In [ ]:
try:
    torch.ao.quantization.quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8, inplace=True)
    evaluate("torch_int8", torch_one)
except Exception as e:
    traceback.print_exc(); RESULTS.append(dict(engine="torch_int8", error=repr(e)[:300]))
del model, wrapped; gc.collect()

## 3. ONNX Runtime fp32 and INT8

In [ ]:
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

def ort_runner(path, threads=THREADS):
    so = ort.SessionOptions(); so.intra_op_num_threads = threads
    sess = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])
    def one(ids):
        x = np.array([ids], dtype=np.int64)
        return sess.run(["probs"], {"input_ids": x, "attention_mask": np.ones_like(x)})[0][0]
    return sess, one

ONNX_INT8 = f"{ONNX_DIR}/model_int8.onnx"
if EXPORT["ok"]:
    try:
        sess, one = ort_runner(ONNX_FP32)
        p = evaluate("onnx_fp32", one)
        RESULTS[-1]["max_prob_diff_vs_torch"] = round(float(np.abs(p - P_REF).max()), 5)
        del sess; gc.collect()
    except Exception as e:
        traceback.print_exc(); RESULTS.append(dict(engine="onnx_fp32", error=repr(e)[:300]))
    try:
        t = time.time()
        quantize_dynamic(ONNX_FP32, ONNX_INT8, weight_type=QuantType.QInt8, use_external_data_format=True)
        print(f"quantized in {time.time() - t:.0f}s")
        sess, one = ort_runner(ONNX_INT8)
        p = evaluate("onnx_int8", one)
        RESULTS[-1]["max_prob_diff_vs_torch"] = round(float(np.abs(p - P_REF).max()), 5)
        del sess; gc.collect()
    except Exception as e:
        traceback.print_exc(); RESULTS.append(dict(engine="onnx_int8", error=repr(e)[:300]))
!du -sh {ONNX_DIR}/* 2>/dev/null

## 4. Pick the demo engine and package it for the Hugging Face Hub
Rule: fastest engine whose log loss stays within **+0.01** of the fp32 reference. Then measure it again with **2 threads** (the free Space has 2 vCPUs).

In [ ]:
res = pd.DataFrame(RESULTS)
display(res); res.to_csv(f"{OUT}/cpu_engines.csv", index=False)

ref = res.loc[res.engine == "torch_fp32", "log_loss"].iloc[0]
ok = res[res["error"].isna()] if "error" in res else res
ok = ok.dropna(subset=["log_loss"])
ok = ok[ok.log_loss <= ref + 0.01].sort_values("p50_ms")
CHOSEN = ok.engine.iloc[0] if len(ok) else "torch_fp32"
print("CHOSEN demo engine:", CHOSEN)

HF = f"{OUT}/hf_model"; shutil.rmtree(HF, ignore_errors=True); os.makedirs(HF)
tok.save_pretrained(HF)
cfg = dict(engine="onnxruntime" if CHOSEN.startswith("onnx") else "torch_int8", source_engine=CHOSEN,
           max_p=INFO["max_p"], max_r=INFO["max_r"], headers=HEADERS, labels=["Response A wins", "Response B wins", "Tie"],
           base_model=INFO.get("model_id"), train_rows=INFO.get("train_rows"), val_log_loss_gpu=INFO.get("val_log_loss"),
           cpu_results=res.fillna("").to_dict("records"))
if CHOSEN.startswith("onnx"):
    src = ONNX_INT8 if CHOSEN == "onnx_int8" else ONNX_FP32
    for f in glob.glob(src + "*"):                               # .onnx + its external data file(s)
        shutil.copy(f, HF)
    cfg["onnx_file"] = os.path.basename(src)
    _, one2 = ort_runner(src, threads=2)
else:                                                            # fallback: ship fp16 weights, quantize at Space startup
    for f in glob.glob(f"{MODEL_DIR}/*"):
        shutil.copy(f, HF)
    m2 = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, dtype=torch.float32, attn_implementation="eager").eval()
    torch.ao.quantization.quantize_dynamic(m2, {torch.nn.Linear}, dtype=torch.qint8, inplace=True)
    torch.set_num_threads(2)
    @torch.inference_mode()
    def one2(ids):
        return torch.softmax(m2(input_ids=torch.tensor([ids]), use_cache=False).logits.float(), -1)[0].numpy()
ms = []
for ids in ROWS[:10]:
    t = time.perf_counter(); one2(ids); ms.append((time.perf_counter() - t) * 1000)
cfg["expected_space_p50_ms_2threads"] = round(float(np.median(ms)), 0)
json.dump(cfg, open(f"{HF}/demo_config.json", "w"), indent=1, default=str)
print("expected latency on a 2-vCPU Space:", cfg["expected_space_p50_ms_2threads"], "ms")
!du -sh {HF}; ls {HF}

## 5. (Optional) push `hf_model/` to the Hugging Face Hub
1. Hugging Face → Settings → Access Tokens → create a **write** token.
2. Kaggle notebook → **Add-ons → Secrets** → add `HF_TOKEN`, and tick it for this notebook.
3. Set `PUSH = True` and your repo name, then run this cell. (Or download `hf_model/` and upload it on the website.)

In [ ]:
PUSH = False
HF_REPO = "YOUR_HF_USERNAME/chatbot-arena-preference-judge"     # <- change
if PUSH:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(HF_REPO, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=f"{OUT}/hf_model", repo_id=HF_REPO, repo_type="model")
    print("uploaded -> https://huggingface.co/" + HF_REPO)